<i>Copyright (c) Recommenders contributors.</i>

<i>Licensed under the MIT License.</i>

# Sequential Recommender Quick Start

### Example: SLi_Rec : Adaptive User Modeling with Long and Short-Term Preferences for Personailzed Recommendation
Unlike a general recommender such as Matrix Factorization or xDeepFM (in the repo) which doesn't consider the order of the user's activities, sequential recommender systems take the sequence of the user behaviors as context and the goal is to predict the items that the user will interact in a short time (in an extreme case, the item that the user will interact next).

This notebook aims to give you a quick example of how to train the [SLi_Rec model](https://www.microsoft.com/en-us/research/uploads/prod/2019/07/IJCAI19-ready_v1.pdf) \[1\] based on a public Amazon dataset.
SLi_Rec \[1\] is a deep learning-based model aims at capturing both long and short-term user preferences for precise recommender systems. To summarize, SLi_Rec has the following key properties:

* It adopts the attentive "Asymmetric-SVD" paradigm for long-term modeling;
* It takes both time irregularity and semantic irregularity into consideration by modifying the gating logic in LSTM.
* It uses an attention mechanism to dynamic fuse the long-term component and short-term component.

In this notebook, we test SLi_Rec on a subset of the public dataset: [Amazon_reviews](http://snap.stanford.edu/data/amazon/productGraph/categoryFiles/reviews_Movies_and_TV_5.json.gz) and [Amazon_metadata](http://snap.stanford.edu/data/amazon/productGraph/categoryFiles/meta_Movies_and_TV.json.gz)

This notebook uses the PyTorch implementation of SLi_Rec. The other sequential models (A2SVD, GRU, Caser, NextItNet, SUM) are covered in [sequential_recsys_amazondataset.ipynb](sequential_recsys_amazondataset.ipynb). 

## 0. Global Settings and Imports

In [ ]:
import os
import sys
import torch

from recommenders.utils.timer import Timer
from recommenders.utils.constants import SEED
from recommenders.datasets.amazon_reviews import download_and_extract, data_preprocessing
from recommenders.models.deeprec.models.sequential.pytorch.sli_rec_pytorch import SLI_RECModel as SeqModel
from recommenders.utils.notebook_utils import store_metadata

print(f"System version: {sys.version}")
print(f"PyTorch version: {torch.__version__}")

#### Parameters

In [ ]:
EPOCHS = 10
BATCH_SIZE = 400
RANDOM_SEED = SEED  # Set None for non-deterministic result

data_path = os.path.join("..", "..", "tests", "resources", "deeprec", "slirec")

##  1. Input data format
The input data contains 8 columns, i.e.,   `<label> <user_id> <item_id> <category_id> <timestamp> <history_item_ids> <history_cateory_ids> <hitory_timestamp>`  columns are seperated by `"\t"`.  item_id and category_id denote the target item and category, which means that for this instance, we want to guess whether user user_id will interact with item_id at timestamp. `<history_*>` columns record the user behavior list up to `<timestamp>`, elements are separated by commas.  `<label>` is a binary value with 1 for positive instances and 0 for negative instances.  One example for an instance is: 

`1       A1QQ86H5M2LVW2  B0059XTU1S      Movies  1377561600      B002ZG97WE,B004IK30PA,B000BNX3AU,B0017ANB08,B005LAIHW2  Movies,Movies,Movies,Movies,Movies   1304294400,1304812800,1315785600,1316304000,1356998400` 

In data preprocessing stage, we have a script to generate some ID mapping dictionaries, so user_id, item_id and category_id will be mapped into interager index starting from 1. And you need to tell the input iterator where is the ID mapping files are. (For example, in the next section, we have some mapping files like user_vocab, item_vocab, and cate_vocab).  The data preprocessing script is at [recommenders/dataset/amazon_reviews.py](../../recommenders/dataset/amazon_reviews.py), you need to call the `_create_vocab(train_file, user_vocab, item_vocab, cate_vocab)` function. Note that ID vocabulary only creates from the train_file, so the new IDs in valid_file or test_file will be regarded as unknown IDs and assigned with a defualt 0 index.

SLi_Rec is time-aware, so it makes use of the `<timestamp>` and `<history_timestamp>` columns to model the time irregularity of the user behaviors.

We use Softmax to the loss function. In training and evalution stage, we group 1 positive instance with `num_ngs` negative instances. Pair-wise ranking can be regarded as a special case of softmax ranking, where `num_ngs` is set to 1. 

More specifically, for training and evalation, you need to organize the data file such that each one positive instance is followed by `num_ngs` negative instances. Our program will take `1+num_ngs` lines as a unit for Softmax calculation. `num_ngs` is a parameter you pass to `fit` and `run_eval`. `train_num_ngs` in `fit` denotes the number of negative instances for training, where a recommended number is 4. `valid_num_ngs` and `num_ngs` in `fit` and `run_eval` denote the number in evaluation. In evaluation, the model calculates metrics among the `1+num_ngs` instances. For the `predict` function, since we only need to calcuate a score for each individual instance, there is no need for `num_ngs` setting.  More details and examples will be provided in the following sections.

For training, you can provide positive instances only and pass `train_num_ngs` to `fit`; the model dynamically samples `train_num_ngs` negatives per positive in each mini-batch.

###  Amazon dataset
Now let's start with a public dataset containing product reviews and metadata from Amazon, which is widely used as a benchmark dataset in recommemdation systems field.

In [ ]:

# for test
train_file = os.path.join(data_path, r'train_data')
valid_file = os.path.join(data_path, r'valid_data')
test_file = os.path.join(data_path, r'test_data')
user_vocab = os.path.join(data_path, r'user_vocab.pkl')
item_vocab = os.path.join(data_path, r'item_vocab.pkl')
cate_vocab = os.path.join(data_path, r'category_vocab.pkl')
output_file = os.path.join(data_path, r'output.txt')
MODEL_DIR = os.path.join(data_path, "model")

reviews_name = 'reviews_Movies_and_TV_5.json'
meta_name = 'meta_Movies_and_TV.json'
reviews_file = os.path.join(data_path, reviews_name)
meta_file = os.path.join(data_path, meta_name)
train_num_ngs = 4 # number of negative instances with a positive instance for training
valid_num_ngs = 4 # number of negative instances with a positive instance for validation
test_num_ngs = 9 # number of negative instances with a positive instance for testing
sample_rate = 0.01 # sample a small item set for training and testing here for fast example

input_files = [reviews_file, meta_file, train_file, valid_file, test_file, user_vocab, item_vocab, cate_vocab]

if not os.path.exists(train_file):
    download_and_extract(reviews_name, reviews_file)
    download_and_extract(meta_name, meta_file)
    data_preprocessing(*input_files, sample_rate=sample_rate, valid_num_ngs=valid_num_ngs, test_num_ngs=test_num_ngs)


#### 1.1 Set model parameters
All parameters are passed explicitly to the model constructor (architecture) and to `fit` (training). `need_sample` is implied: the training file holds positives only and `train_num_ngs` negatives are sampled in-batch. `train_num_ngs`, `valid_num_ngs` and `num_ngs` (in `fit`/`run_eval`) set the 1-positive-to-N-negatives grouping used by the softmax loss and the ranking metrics.

## 2. Create model
When both hyper-parameters and data iterator are ready, we can create a model:

In [ ]:
model = SeqModel(
    user_vocab=user_vocab,
    item_vocab=item_vocab,
    cate_vocab=cate_vocab,
    item_embedding_dim=32,
    cate_embedding_dim=8,
    user_embedding_dim=16,
    hidden_size=40,
    attention_size=40,
    max_seq_length=50,
    layer_sizes=[100, 64],
    att_fcn_layer_sizes=[80, 40],
    dropout=[0.3, 0.3],
    seed=RANDOM_SEED,
)

## to load a pre-trained model instead of training from scratch:
# model.load_model(os.path.join(MODEL_DIR, "best_model"))

Now let's see what is the model's performance at this point (without starting training):

In [ ]:
# test_num_ngs is the number of negative lines after each positive line in your test_file
print(model.run_eval(test_file, num_ngs=test_num_ngs)) 

AUC=0.5 is a state of random guess. We can see that before training, the model behaves like random guessing.

#### 2.1 Train model
Next we want to train the model on a training set, and check the performance on a validation dataset. Training the model is as simple as a function call:

In [ ]:
with Timer() as train_time:
    model = model.fit(
        train_file,
        valid_file,
        epochs=EPOCHS,
        batch_size=BATCH_SIZE,
        learning_rate=0.001,
        train_num_ngs=train_num_ngs,
        valid_num_ngs=valid_num_ngs,
        embed_l2=0.0,
        layer_l2=0.0,
        show_step=20,
        save_model=True,
        model_dir=MODEL_DIR,
    )

# valid_num_ngs is the number of negative lines after each positive line in valid_file
# we evaluate on valid_file every epoch
print('Time cost for training is {0:.2f} mins'.format(train_time.interval/60.0))

#### 2.2  Evaluate model

Again, let's see what is the model's performance now (after training):

In [ ]:
res_syn = model.run_eval(test_file, num_ngs=test_num_ngs)
print(res_syn)


In [ ]:
# Record results for tests - ignore this cell
store_metadata("auc", res_syn["auc"])
store_metadata("logloss", res_syn["logloss"])
store_metadata("mean_mrr", res_syn["mean_mrr"])
store_metadata("ndcg@2", res_syn["ndcg@2"])
store_metadata("ndcg@4", res_syn["ndcg@4"])
store_metadata("ndcg@6", res_syn["ndcg@6"])
store_metadata("group_auc", res_syn["group_auc"])


If we want to get the full prediction scores rather than evaluation metrics, we can do this:

In [ ]:
model = model.predict(test_file, output_file)

In [ ]:
# The data was downloaded in tmpdir folder. You can delete them manually if you do not need them any more.

#### 2.3  Running SLi_Rec with large dataset
Here is the performance of SLi_Rec using the whole amazon dataset with 1,697,533 positive instances.
<br>Settings for reproducing the results:
<br>`learning_rate=0.001, dropout=0.3, item_embedding_dim=32, cate_embedding_dim=8, l2_norm=0, batch_size=400, 
train_num_ngs=4, valid_num_ngs=4, test_num_ngs=49`


We compare the running time with CPU only and with GPU on the larger dataset. It appears that GPU can significantly accelerate the training. Hardware specification for running the large dataset: 
<br>GPU: Tesla P100-PCIE-16GB
<br>CPU: 6 cores Intel(R) Xeon(R) CPU E5-2690 v4 @ 2.60GHz
 
| Models | AUC | g-AUC | NDCG@2 | NDCG@10 | seconds per epoch on GPU | seconds per epoch on CPU| config |
| :------| :------: | :------: | :------: | :------: | :------: | :------: | :------ |
| SLi_Rec | 0.8631 | 0.8519 | 0.3491 | 0.4842 | 549.6 | 5014.0 | attention_size=40, max_seq_length=50, hidden_size=40|

 Note 1: These reference numbers were obtained with the original TensorFlow implementation of SLi_Rec, grid searched with a coarse granularity, and are for reference only.
 <br>Note 2: A comparison with the other sequential models on the same dataset is available in [sequential_recsys_amazondataset.ipynb](sequential_recsys_amazondataset.ipynb).

## 3. Loading Trained Models
In this section, we provide a simple example to illustrate how we can use the trained model to serve for production demand.

Suppose we are in a new session. First let's load a previous trained model:

In [ ]:
model_best_trained = SeqModel(
    user_vocab=user_vocab,
    item_vocab=item_vocab,
    cate_vocab=cate_vocab,
    item_embedding_dim=32,
    cate_embedding_dim=8,
    user_embedding_dim=16,
    hidden_size=40,
    attention_size=40,
    max_seq_length=50,
    layer_sizes=[100, 64],
    att_fcn_layer_sizes=[80, 40],
    dropout=[0.3, 0.3],
    seed=RANDOM_SEED,
)
path_best_trained = os.path.join(MODEL_DIR, "best_model")
print('loading saved model in {0}'.format(path_best_trained))
model_best_trained.load_model(path_best_trained)

Let's see if we load the model correctly. The testing metrics should be close to the numbers we have in the training stage.

In [ ]:
model_best_trained.run_eval(test_file, num_ngs=test_num_ngs)

And we make predictions using this model. In the next step, we will make predictions using a serving model. Then we can check if the two result files are consistent.

In [ ]:
model_best_trained.predict(test_file, output_file)

## References
\[1\] Zeping Yu, Jianxun Lian, Ahmad Mahmoody, Gongshen Liu, Xing Xie. Adaptive User Modeling with Long and Short-Term Preferences for Personailzed Recommendation. In Proceedings of the 28th International Joint Conferences on Artificial Intelligence, IJCAI’19, Pages 4213-4219. AAAI Press, 2019.